# Encoder × Segmentor Complexity Comparison
Measures **params**, **model size (MB)**, **inference latency (ms/window)**, **throughput (fps)**, and **peak memory** for all 12 encoder–segmentor combinations.

In [ ]:
import sys, os, time, copy, tempfile
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
WINDOW_SIZE = 120
N_WARMUP    = 10
N_RUNS      = 100

print(f'Device: {DEVICE}')

In [ ]:
# ── Encoder builders ────────────────────────────────────────────────────────

def build_identity():
    from models.encoders.Identity import IdentityEncoder
    enc = IdentityEncoder(feature_dim=33 * 3)   # MPW: 33 joints × 3 dims
    return enc, 33 * 3                           # (encoder, input_feature_dim)


def build_gcn():
    from models.encoders.GCN.MS_GCN_ENCODER import MSGCNEncoder
    enc = MSGCNEncoder(
        graph_args={'layout': 'mp', 'strategy': 'spatial'},
        num_joints=33,
        in_channels=3,
        filters=64,
        dil=[1, 2, 4, 8, 16, 32, 64, 128, 256, 512],
    )
    return enc, 33 * 3


def build_mamp():
    from models.encoders.MY_MAMP.encoder import MAMPFeatureEncoder
    enc = MAMPFeatureEncoder(
        skeleton_type='world_mp_cropped_iou',
        checkpoint_path='/data2/pathml/MAMP/checkpoints/checkpoints/ntu120_xset.pth',
        config_path='/code/jjiang23/pathml/aim2_balanceV2/models/encoders/MY_MAMP/pretrain_mamp_t120_layer8+5_mask90.yaml',
    )
    return enc, 33 * 3


def build_mae():
    from models.encoders.MY_SKELETONMAE.encoder import MAEFeatureEncoder
    enc = MAEFeatureEncoder(
        skeleton_type='world_mp_cropped_iou',
        checkpoint_path='/data2/pathml/MAMP/checkpoints/checkpoints/mae-checkpoint.pth',
        config_path='/code/jjiang23/pathml/aim2_balanceV2/models/encoders/MY_SKELETONMAE/pretrain_mae_t120_layer8+5_mask90.yaml',
    )
    return enc, 33 * 3


ENCODERS = {
    'Identity': build_identity,
    'GCN':      build_gcn,
    'MAMP':     build_mamp,
    'MAE':      build_mae,
}

In [ ]:
# ── Segmentor builders  (all take enc_out_dim as only positional arg) ────────

def build_mstcn(enc_out_dim):
    from models.segmentors.MSTCNplus import MS_TCN2
    return MS_TCN2(
        num_layers_PG=11,
        num_layers_R=10,
        num_R=3,
        num_f_maps=64,
        dim=enc_out_dim,
        num_classes=2,
    )


def build_bilstm(enc_out_dim):
    from models.segmentors.biLSTM import BiLSTM
    return BiLSTM(
        dim=enc_out_dim,
        num_classes=2,
        hidden_size=128,
        num_layers=2,
        dropout=0.3,
    )


def build_asformer(enc_out_dim):
    from models.segmentors.ASFormer import ASFormer
    return ASFormer(
        dim=enc_out_dim,
        num_classes=2,
        num_f_maps=64,
        num_layers=10,
        num_decoders=3,
        r1=2,
        r2=2,
        channel_masking_rate=0.3,
        att_type='block_att',
    )


SEGMENTORS = {
    'MS-TCN2':  build_mstcn,
    'BiLSTM':   build_bilstm,
    'ASFormer': build_asformer,
}

In [ ]:
# ── Measurement helpers ──────────────────────────────────────────────────────

def count_params(module):
    total     = sum(p.numel() for p in module.parameters())
    trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
    return total, trainable


def model_size_mb(module):
    """Serialize state dict to temp file, return size in MB."""
    with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as tmp:
        torch.save(module.state_dict(), tmp.name)
        size = os.path.getsize(tmp.name) / 1e6
    os.unlink(tmp.name)
    return size


@torch.no_grad()
def benchmark(encoder, segmentor, input_feat_dim, device,
               window_size=WINDOW_SIZE, n_warmup=N_WARMUP, n_runs=N_RUNS):
    """
    Returns mean latency (ms), std, throughput (fps), peak GPU memory (MB).
    Input: (1, window_size, input_feat_dim) — matches PoseDataset output.
    """
    encoder.eval().to(device)
    segmentor.eval().to(device)
    x = torch.randn(1, window_size, input_feat_dim, device=device)

    # Warmup
    for _ in range(n_warmup):
        feat = encoder(x)
        segmentor(feat)

    latencies = []

    if device.type == 'cuda':
        torch.cuda.reset_peak_memory_stats(device)
        starts = [torch.cuda.Event(enable_timing=True) for _ in range(n_runs)]
        ends   = [torch.cuda.Event(enable_timing=True) for _ in range(n_runs)]
        for i in range(n_runs):
            starts[i].record()
            feat = encoder(x)
            segmentor(feat)
            ends[i].record()
        torch.cuda.synchronize()
        latencies = [s.elapsed_time(e) for s, e in zip(starts, ends)]
        peak_mem = torch.cuda.max_memory_allocated(device) / 1e6
    else:
        for _ in range(n_runs):
            t0 = time.perf_counter()
            feat = encoder(x)
            segmentor(feat)
            t1 = time.perf_counter()
            latencies.append((t1 - t0) * 1000)
        peak_mem = None

    mean_ms = float(np.mean(latencies))
    fps     = (window_size / mean_ms) * 1000   # frames processed per second
    return mean_ms, float(np.std(latencies)), fps, peak_mem

In [ ]:
# ── Main loop ────────────────────────────────────────────────────────────────
rows = []

for enc_name, enc_builder in ENCODERS.items():
    # ── Try to build encoder ──────────────────────────────────────────────
    try:
        encoder, input_feat_dim = enc_builder()
        enc_total, enc_trainable = count_params(encoder)
        enc_size_mb = model_size_mb(encoder)
        enc_ok = True
        print(f'\n{enc_name}: {enc_total:,} params  ({enc_size_mb:.1f} MB)  out_dim={encoder.out_dim}')
    except Exception as e:
        print(f'\n{enc_name}: FAILED to load — {e}')
        enc_ok = False

    for seg_name, seg_builder in SEGMENTORS.items():
        label = f'{enc_name}+{seg_name}'
        if not enc_ok:
            rows.append({'combo': label, 'encoder': enc_name, 'segmentor': seg_name,
                         'status': 'encoder_unavailable'})
            continue

        try:
            segmentor = seg_builder(encoder.out_dim)
            seg_total, seg_trainable = count_params(segmentor)
            seg_size_mb = model_size_mb(segmentor)

            total_params    = enc_total + seg_total
            trainable_params = enc_trainable + seg_trainable
            total_size_mb   = enc_size_mb + seg_size_mb

            mean_ms, std_ms, fps, peak_mem = benchmark(
                encoder, segmentor, input_feat_dim, DEVICE
            )

            print(f'  {seg_name:<10} seg={seg_total:>8,} total={total_params:>10,}  '
                  f'{mean_ms:.2f}±{std_ms:.2f}ms  {fps:.0f}fps  '
                  f'{total_size_mb:.1f}MB  '
                  + (f'mem={peak_mem:.1f}MB' if peak_mem else 'mem=CPU'))

            rows.append({
                'combo':            label,
                'encoder':          enc_name,
                'segmentor':        seg_name,
                'status':           'ok',
                'enc_params':       enc_total,
                'enc_trainable':    enc_trainable,
                'seg_params':       seg_total,
                'seg_trainable':    seg_trainable,
                'total_params':     total_params,
                'trainable_params': trainable_params,
                'enc_size_mb':      enc_size_mb,
                'seg_size_mb':      seg_size_mb,
                'total_size_mb':    total_size_mb,
                'latency_ms':       mean_ms,
                'latency_std_ms':   std_ms,
                'throughput_fps':   fps,
                'peak_mem_mb':      peak_mem,
            })

        except Exception as e:
            print(f'  {seg_name:<10} FAILED — {e}')
            rows.append({'combo': label, 'encoder': enc_name, 'segmentor': seg_name,
                         'status': f'failed: {e}'})

df = pd.DataFrame(rows)
df_ok = df[df['status'] == 'ok'].copy()

In [ ]:
# ── Summary table ────────────────────────────────────────────────────────────
cols = ['combo', 'enc_params', 'seg_params', 'total_params', 'trainable_params',
        'total_size_mb', 'latency_ms', 'latency_std_ms', 'throughput_fps', 'peak_mem_mb']
display_cols = [c for c in cols if c in df_ok.columns]

fmt = {
    'enc_params':       '{:,.0f}',
    'seg_params':       '{:,.0f}',
    'total_params':     '{:,.0f}',
    'trainable_params': '{:,.0f}',
    'total_size_mb':    '{:.1f}',
    'latency_ms':       '{:.2f}',
    'latency_std_ms':   '{:.2f}',
    'throughput_fps':   '{:.0f}',
    'peak_mem_mb':      '{:.1f}',
}

df_display = df_ok[display_cols].set_index('combo')
df_display.style.format(fmt, na_rep='—')

In [ ]:
# ── Bar charts ───────────────────────────────────────────────────────────────
if df_ok.empty:
    print('No successful combinations to plot.')
else:
    metrics = [
        ('total_params',     'Total Parameters',     'params',  False),
        ('trainable_params', 'Trainable Parameters', 'params',  False),
        ('total_size_mb',    'Model Size on Disk',   'MB',      False),
        ('latency_ms',       'Inference Latency',    'ms / window', False),
        ('throughput_fps',   'Throughput',           'frames / sec', True),
        ('peak_mem_mb',      'Peak GPU Memory',      'MB',      False),
    ]
    metrics = [(m, t, u, h) for m, t, u, h in metrics
               if m in df_ok.columns and df_ok[m].notna().any()]

    n_metrics = len(metrics)
    ncols = 2
    nrows = (n_metrics + 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4.5 * nrows))
    axes = axes.flatten()

    colors = plt.cm.tab10(np.linspace(0, 0.9, len(df_ok)))

    for ax, (metric, title, unit, higher_better) in zip(axes, metrics):
        vals = df_ok.set_index('combo')[metric]
        bars = ax.barh(vals.index, vals.values, color=colors)
        ax.set_title(title)
        ax.set_xlabel(unit)
        ax.xaxis.set_major_formatter(ticker.FuncFormatter(
            lambda x, _: f'{x:,.0f}' if x >= 1000 else f'{x:.2g}'
        ))
        ax.invert_yaxis()
        # Annotate bars
        for bar, val in zip(bars, vals.values):
            ax.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height() / 2,
                    f'{val:,.0f}' if val >= 100 else f'{val:.2f}',
                    va='center', fontsize=8)
        note = '▲ higher is better' if higher_better else '▼ lower is better'
        ax.set_title(f'{title}  ({note})', fontsize=10)

    # Hide unused axes
    for ax in axes[n_metrics:]:
        ax.set_visible(False)

    plt.suptitle(
        f'Encoder × Segmentor Physical Performance  '
        f'[device={DEVICE}, window={WINDOW_SIZE}f, n={N_RUNS} runs]',
        fontsize=12, y=1.01
    )
    plt.tight_layout()
    plt.savefig('../results/model_complexity.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Stacked encoder vs segmentor param breakdown ─────────────────────────────
if not df_ok.empty and 'enc_params' in df_ok.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    combos = df_ok['combo'].values
    y = np.arange(len(combos))

    enc_trainable_vals = df_ok['enc_trainable'].values
    enc_frozen_vals    = df_ok['enc_params'].values - enc_trainable_vals
    seg_vals           = df_ok['seg_params'].values

    ax.barh(y, enc_frozen_vals,  label='Encoder (frozen)',   color='#4e79a7', alpha=0.7)
    ax.barh(y, enc_trainable_vals, left=enc_frozen_vals,
            label='Encoder (trainable)', color='#4e79a7')
    ax.barh(y, seg_vals, left=df_ok['enc_params'].values,
            label='Segmentor (trainable)', color='#f28e2b')

    ax.set_yticks(y)
    ax.set_yticklabels(combos)
    ax.invert_yaxis()
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M' if x >= 1e6 else f'{x/1e3:.0f}K'))
    ax.set_xlabel('Parameters')
    ax.set_title('Parameter Breakdown: Encoder (frozen vs trainable) + Segmentor')
    ax.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig('../results/model_param_breakdown.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── Save results CSV ─────────────────────────────────────────────────────────
os.makedirs('../results', exist_ok=True)
df.to_csv('../results/model_complexity.csv', index=False)
print('Saved → results/model_complexity.csv')
print(f'Successful combinations: {len(df_ok)} / {len(df)}')
if not df_ok.empty:
    print('\nFastest:  ', df_ok.loc[df_ok['latency_ms'].idxmin(), 'combo'])
    print('Lightest: ', df_ok.loc[df_ok['total_params'].idxmin(), 'combo'])
    print('Smallest: ', df_ok.loc[df_ok['total_size_mb'].idxmin(), 'combo'])